## FIle dedicato all'addestramento del modello
Il file contiene le operazioni di addestramento necessarie, prende in ingresso gli iperparametri e fornisce in uscita le metriche di performance ottenute.
Si utilizza la rete ResNet-18 già addestrata.
$$
\quad
$$
Come primo step si costruisce la funzione che consente di inizializzare la rete: riceve in ingresso gli iperparametri e procede a caricare il modello, sbloccando solo il numero di layer passati in ingresso.


In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import balanced_accuracy_score

def initial_reset(num_classes, training_layers=0):
  """
  Inizializza la ResNet 18  pre-addestrata
  training_layers rappresenta il numero di layer da sbloccare a partire
  dalla fine (1 si sblocca il layer 4 e il layer di output, 0 solo il layer
   di output etc...)
  """
  model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
  #inizialmente si blocca tutta la rete
  for param in model.parameters():
    param.requires_grad=False
  available_layers=[model.layer4, model.layer3, model.layer2, model.layer1]
  unlocked_layers=min(training_layers, len(available_layers))
  for i in range(unlocked_layers):
    for param in available_layers[i].parameters():
      param.requires_grad=True
  #sostituzione ultimo layer per adattarlo al numero di classi passate
  n_features = model.fc.in_features
  model.fc=nn.Linear(n_features, num_classes)
  return model



Si passa ora alla definizione del metodo di training: il metodo prende in ingresso il modello da allenare, il numero di epoche, il dataloader(contenente la divisione in training e validation) e l'optimizer.
<br>
La metrica di loss scelta è la Categorical Cross Entropy e si mantiene un dizionario che manterrà le statistiche relative alle performance (loss di training, loss di validation e accuracy). Si utilizza un learning rate decadente che viene fornito dallo scheduler (preso in ingresso).
<br>
Il parametro class_weights è un tensore che da un 'peso' a ciascuna classe: più la classe è rara maggiore sarà il peso ad essa associato, questo peso influirà sulla loss.
- Se `class_weights=None` allora la Categorical Cross Entropy sarà:
$$
 -log(p_c)
$$

- Se `class_weights != None` allora la Categorical Cross Entropy sarà:

$$
 -w_clog(p_c)
$$
Dove $w_c$ è il peso passato dal tensore in ingresso, mentre $p_c$ è la probabilità che la rete neurale ha assegnato alla classe corretta.


In [ ]:


def train_model_decay_lr(model, dataloaders, optimizer, epochs,scheduler=None, class_weights=None):
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  model = model.to(device)
  if class_weights is not None:
    class_weights=class_weights.to(device)
    loss_function=nn.CrossEntropyLoss(class_weights)
  else:
    loss_function= nn.CrossEntropyLoss()
  best_acc=0.0
  history={'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_bal_acc':[],'val_bal_acc':[] }
  best_model=copy.deepcopy(model.state_dict()) #si salva il modello migliore (inizialmente è la rete non allenata)
  for epoch in range(epochs):
    print(f'\nEpoca {epoch+1}/{epochs}')
    for phase in ['train', 'val']:
      if phase == 'train':
        model.train() #modalità di training
      else:
        model.eval() #modalità d inferenza
      curr_loss=0.0
      curr_correct=0
      all_preds=[]
      all_labels=[]
      for inputs, labels in dataloaders[phase]:
        inputs=inputs.to(device)
        labels=labels.to(device)
        optimizer.zero_grad() #si azzera il gradiente per non mescolare gli errori
        #Passo forward
        with torch.set_grad_enabled(phase == 'train'): #si memorizza il gradiente solo se in training
          outputs=model(inputs)
          _,preds= torch.max(outputs,1)
          loss=loss_function(outputs,labels)
          #Passo backward (disponibile solo se in training)
          if phase=='train':
            loss.backward()
            optimizer.step()
        curr_loss+=loss.item()*inputs.size(0)
        curr_correct+=torch.sum(preds==labels.data)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
      epoch_loss=curr_loss/len(dataloaders[phase].dataset)
      epoch_acc=curr_correct.double()/len(dataloaders[phase].dataset)
      epoch_bal_acc=balanced_accuracy_score(all_labels, all_preds)
      print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}| Bal. Acc: {epoch_bal_acc:.4f}')
      history[f'{phase}_loss'].append(epoch_loss)
      history[f'{phase}_acc'].append(epoch_acc.item())
      history[f'{phase}_bal_acc'].append(epoch_bal_acc)
      if phase=='val' and epoch_bal_acc>best_acc:
        best_acc=epoch_bal_acc
        best_model=copy.deepcopy(model.state_dict())
    if scheduler is not None:
      scheduler.step()
  print(f'Miglior Accuratezza (Validation): {best_acc:.4f}')
  model.load_state_dict(best_model)
  return model, history

Metodo di utility per sviluppare i grafici relativi alla loss e all'accuratezza.

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(history):
    """
    Prende in ingresso il dizionario 'history' generato da train_model
    e plotta i grafici di Loss e Accuratezza.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    #Figura con due sotto-grafici (1 riga, 2 colonne)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    # Grafico per la Loss (Errore)
    ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    ax1.set_title('Andamento della Loss')
    ax1.set_xlabel('Epoche')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    # Grafico per l'Accuratezza
    ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
    ax2.set_title('Andamento dell\'Accuratezza')
    ax2.set_xlabel('Epoche')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    #Grafico per la Balanced Accuracy
    ax3.plot(epochs, history['train_bal_acc'], 'b-', label='Training Bal. Acc.')
    ax3.plot(epochs, history['val_bal_acc'], 'r-', label='Validation Bal. Acc.')
    ax3.set_title('Andamento della Balanced Accuracy')
    ax3.set_xlabel('Epoche')
    ax3.set_ylabel('Balanced Accuracy')
    ax3.legend()
    ax3.grid(True)

    plt.tight_layout()
    plt.show()

# Early Stopping e Addestramento Avanzato

In [ ]:
class EarlyStopping:
    """Interrompe l'addestramento se la validation loss non migliora dopo un tot di epoche."""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience     # Quante epoche aspettare prima di fermarsi
        self.min_delta = min_delta   # Miglioramento minimo richiesto per non resettare il contatore
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"-> EarlyStopping Counter: {self.counter} su {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def train_model_advanced(model, dataloaders, optimizer, epochs, scheduler=None, class_weights=None, patience=5):
    """Ciclo di addestramento avanzato con supporto a Early Stopping e ReduceLROnPlateau."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    if class_weights is not None:
        class_weights = class_weights.to(device)
        loss_function = nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_function = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_bal_acc':[], 'val_bal_acc':[] }
    best_model = copy.deepcopy(model.state_dict())

    # Inizializziamo il controllore dell'Early Stopping
    early_stopper = EarlyStopping(patience=patience, min_delta=0.001)

    for epoch in range(epochs):
        print(f'\nEpoca {epoch+1}/{epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            curr_loss = 0.0
            curr_correct = 0
            all_preds = []
            all_labels = []

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = loss_function(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                curr_loss += loss.item() * inputs.size(0)
                curr_correct += torch.sum(preds == labels.data)

                # Vettori estratti correttamente per scikit-learn
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = curr_loss / len(dataloaders[phase].dataset)
            epoch_acc = curr_correct.double() / len(dataloaders[phase].dataset)
            epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Bal. Acc: {epoch_bal_acc:.4f}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_bal_acc'].append(epoch_bal_acc)

            # Logiche di controllo eseguite SOLO a fine validazione
            if phase == 'val':
                # 1. Salva i pesi se è la loss minore mai vista
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_model = copy.deepcopy(model.state_dict())

                # 2. Aggiorna lo scheduler (passando la validation loss se è un ReduceLROnPlateau)
                if scheduler is not None:
                    old_lr = optimizer.param_groups[0]['lr']
                    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                        scheduler.step(epoch_loss)
                    else:
                        scheduler.step()
                    new_lr = optimizer.param_groups[0]['lr']
                    if new_lr < old_lr:
                        print(f"Learning Rate decresciuto automaticamente a: {new_lr}")

                # 3. Interroga l'Early Stopping
                early_stopper(epoch_loss)

        # Interrompe l'addestramento se la loss ha smesso di scendere
        if early_stopper.early_stop:
            print(f"Early Stopping attivato! Addestramento interrotto anticipatamente.")
            break

    print(f'\nAddestramento concluso. Ripristino dei pesi migliori (Miglior Validation Loss: {best_val_loss:.4f})')
    model.load_state_dict(best_model)
    return model, history